# 任务2 · 选题A：基础 RAG 图书馆问答

**程序演示工程师的工作**：把程序调通，运行对照实验，现场用 1—1.5 分钟展示检索依据和回答。完整操作见 `docs/新手操作与分工说明.md`。

本项目使用**虚构星河大学图书馆资料**；不是任何真实学校的制度。真实模型运行前没有预填答案。

必须覆盖：① 简单段落 vs overlap 切分；② 温度 0 vs 1；③ 展示检索片段及 RAG 回答。另加定长无重叠组，帮助区分片段长度和重叠带来的影响。

## 1. 检查当前目录与 Python

选中代码格按 **Shift + Enter**，等待左边运行标志结束。首次先按新手说明安装依赖和下载模型；课堂开始前完成模型加载。

In [ ]:
from pathlib import Path
import sys, json, html
from IPython.display import display, HTML

ROOT = Path.cwd()
assert (ROOT / "rag_core.py").is_file(), "请从项目文件夹启动 Notebook，不要只复制这一个文件。"
print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目目录:", ROOT)
from rag_core import load_documents, split_documents, prepare_models, make_system
from experiments import run_experiments, render_record


## 2. 看懂完整流程

**离线准备**：知识资料 → 切分 → BGE 向量 → Chroma 保存。

**在线问答**：问题 → 同一个 BGE 模型 → 找最相关的两个片段 → 组成提示词 → Qwen 生成回答。

这里的“在线”指实时处理问题，不表示必须联网。模型下载完成后，问答完全在本机执行。我们使用已训练模型，不需要自己训练。

In [ ]:
documents = load_documents(ROOT / "data" / "knowledge.txt")
print("资料数量：", len(documents))
print(documents[0].text[:380])

for strategy in ("paragraph", "fixed", "overlap"):
    chunks = split_documents(documents, strategy, chunk_size=240, overlap=80)
    print(f"{strategy}: {len(chunks)} 块，平均 {sum(len(c.text) for c in chunks)/len(chunks):.1f} 字符")
print("注意：paragraph按空行分段，不限制为240字符；fixed与overlap才使用240字符窗口。")


## 3. 观察切分与重叠（此步不需要模型）

`chunk_size=240` 表示每块最多240个字符；`overlap=80` 表示下一块从前一块末尾向前80字符开始。课件也可按 token 切分，本演示为了直观使用字符。相邻重复部分会增加存储和计算，不保证问答一定更好。

In [ ]:
chunks = split_documents(documents, "overlap", chunk_size=240, overlap=80)
for chunk in chunks[:2]:
    print(f"原文位置 [{chunk.start}:{chunk.end}] 来源 {chunk.source}\n{chunk.text}\n")
print("相邻重复部分：")
print(chunks[0].text[-80:])
assert chunks[0].text[-80:] == chunks[1].text[:80]


## 4. 准备真实模型（课前执行）

第一次需要联网下载，合计约1.1GB的模型文件，依赖和CPU运行还需要额外空间和内存。已经下载且校验通过会复用本地文件。

若下载站无法连接，此处会报错并停止；**不会产生假答案**。不要把这格放到课堂倒计时内运行。

In [ ]:
manifest = prepare_models()
for name, info in manifest.items():
    print(name, info["repo_id"], "版本", info["revision"])


## 5. 建立三套索引并加载生成模型（课前执行）

Chroma 是嵌入式向量库，无需安装独立服务器。每套索引使用相同的 Embedding 模型。这里只建立本项目的集合，不清空其他数据库。

In [ ]:
system = make_system()
for strategy in ("paragraph", "fixed", "overlap"):
    print(strategy, system.build_index(strategy), "个片段已就绪")
system.models.load_generator()
print("模型已加载。现在可以进入现场问答。")


## 6. 现场演示：先展示检索，再让模型回答

**上台主要操作这两格。** 先向老师指出检索来源，再生成一次回答。相似度是相关程度，不是答案正确率。

默认选简单事实题，减少现场耗时；其他问题与综合实验已放在后面。

In [ ]:
question = "本科生最多同时借几册纸质图书？每册借期多久？"
hits = system.retrieve(question, strategy="overlap", top_k=2)
print("问题：", question)
for i, hit in enumerate(hits, 1):
    print(f"\n[S{i}] {hit['source']} | 余弦相似度 {hit['score']:.4f}\n{hit['text']}")


In [ ]:
live_result = system.answer(question, strategy="overlap", temperature=0, seed=42,
                            top_k=2, max_new_tokens=96, hits=hits)
display(HTML(render_record(live_result)))
print("请对照原文人工核对答案；模型回答不等于事实保证。")


## 7. 完成两类对照实验（主要在课前运行）

**切分对照**：相同问题，三种切分，温度均为0，Top-K均为2。

**温度对照**：复用 overlap 检索出的同一份上下文；温度0跑两次，温度1用三个不同种子各跑一次。温度0使用贪心解码，温度1开启采样，其他采样限制一致。

快速模式仅测试一个综合问题。正式报告把 `RUN_FULL` 改为 `True`，覆盖直接事实、综合问题、相似主题和资料不足四类问题。运行耗时依电脑而异，课堂不要点“全部运行”。

In [ ]:
RUN_FULL = False  # 正式收集实验数据前改为 True；第一次可先快速预演。
questions = json.loads((ROOT / "data" / "questions.json").read_text(encoding="utf-8"))
report_dir = run_experiments(system, questions, quick=not RUN_FULL)
print("本次真实实验目录：", report_dir)
print("完整浏览器报告：", report_dir / "report.html")


## 8. 查看对比结果

HTML、CSV、JSON 全部来自本次真实运行。下表只显示概览；完整检索依据和提示词保存在 JSON，网页报告可直接在浏览器打开。

**如出现长度上限提醒，回答可能被截断**。对该问题可使用 `system.answer(..., max_new_tokens=160)` 重试；重新比较时双方都要用同一上限并记录设置。

In [ ]:
records = json.loads((report_dir / "records.json").read_text(encoding="utf-8"))
# 报告可能含顶层元数据；兼容以 records 保存的结构。
if isinstance(records, dict):
    records = records["records"]
rows = []
for r in records:
    vals = [r.get("question_id", ""), r.get("experiment", ""), r["strategy"],
            str(r["temperature"]), str(r["seed"]), r["answer"]]
    rows.append("<tr>" + "".join("<td>"+html.escape(v)+"</td>" for v in vals) + "</tr>")
display(HTML("<table border='1'><tr><th>问题</th><th>实验</th><th>切分</th><th>温度</th><th>种子</th><th>真实回答</th></tr>" + "".join(rows) + "</table>"))


## 9. 人工评价：把观察交给组员

不要提前写“overlap一定更好”“温度1一定更差”。请按真实结果填写：

| 要检查什么 | 你记录什么 |
|---|---|
| 检索是否找到了关键事实 | 片段编号、原文、是否漏掉条件 |
| 回答是否正确完整 | 对照 `data/knowledge.txt`，标记错误、遗漏、编造 |
| 段落与overlap有何不同 | 检索内容、答案和上下文长度；不同切分有不同成本 |
| fixed与overlap有何不同 | 相同窗口长度下的实际变化；Top-K相同不等于有效信息量相同 |
| 温度0与1是否有变化 | 相同提示词下多次输出；相同也如实记录 |
| 资料不足时如何回答 | 检查馆长私人手机号问题，不能猜号码 |

这里只是小规模教学实验，不能用四个问题宣称整个RAG系统准确率已经很高。填写模板见 `docs/实验记录模板.md`。

## 10. 你最终要交什么

- 本项目的完整代码、Notebook、依赖文件及资料。
- `results/` 中一套**完整的真实实验结果**和人工分析。
- 约1—1.5分钟的现场讲稿，见 `docs/课堂演示与答辩.md`。
- 使用最终演示电脑做一次预演、录制备用视频并复制到U盘。

课前先加载模型和索引，现场展示第6部分，并解释第8部分的真实对照结果。讲台Win10没有本项目的环境，优先用自己的电脑接HDMI。

**诚实说明局限**：0.5B小模型可能遗漏条件或编造引用；低温度也不能消除幻觉。模型是否运行成功和答案是否正确是两项不同的检查。